# Shared Urban + depth: first runtime prototype

Revision **PARKING-SHARED-NANO-2026-10-27-r1**, latest 27 October 2026. Run top to bottom on a **CPU runtime**. No GPU, dataset staging, training, KD, old-runtime restore or phone installation.

One Urban Nano backbone/neck feeds pretrained detection/corner predictions and a new small depth decoder. **The depth decoder is UNTRAINED: its output must not be used as distance.** This notebook checks one-pass sharing, frozen BatchNorm, finite depth gradients, ONNX parity and host CPU speed. It creates a synthetic camera-OFF phone-test package; iPhone latency remains unmeasured.

The existing models and the 64-image evaluation set stay unchanged. Only the exact hash-verified official Urban checkpoint and clean pinned upstream source are loaded. Upstream Urban/Ultralytics license terms still apply.

## 1. Mount Drive and update project main
Use a fresh CPU Colab session. The run directory is timestamped so completed runs are preserved. Rerun later cells to resume this run; rerunning this setup cell creates a new run. If the new script is not yet published to main, ask for publication rather than substituting an older notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, os, shlex, subprocess, sys, zipfile
from google.colab import drive, files
drive.mount('/content/drive')
REVISION='PARKING-SHARED-NANO-2026-10-27-r1'  # Latest 27 October 2026
PROJECT=Path('/content/mobile_adas3d')
URL='/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch','main','--single-branch',URL,str(PROJECT)],check=True)
else:
    branch=subprocess.check_output(['git','-C',str(PROJECT),'branch','--show-current'],text=True).strip()
    if branch!='main': raise RuntimeError(f'Expected main, found {branch}; preserve the checkout')
    for command in (['diff','--quiet'],['diff','--cached','--quiet']):
        if subprocess.run(['git','-C',str(PROJECT),*command]).returncode: raise RuntimeError('Preserve tracked changes before updating')
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)
RUN_ID='parking_shared_nano_runtime_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
BASE=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/parking_shared_nano')
OUTPUT=BASE/RUN_ID
SOURCE=Path('/content/UrbanShared_caa8537')
WEIGHTS=BASE/'pinned_checkpoints'
SCRIPT=PROJECT/'scripts/screen_parking_shared_model.py'
CONFIG=PROJECT/'configs/parking_shared_nano_v1.json'
for path in [SCRIPT,CONFIG,PROJECT/'models/parking_shared.py',PROJECT/'tests/test_parking_shared_model.py']:
    if not path.is_file(): raise FileNotFoundError(f'{path} missing; this workflow must be published to main first')
if json.loads(CONFIG.read_text())['revision']!=REVISION: raise RuntimeError('Notebook and project revisions differ')
def run_logged(command,name):
    command=[str(x) for x in command]
    logs=OUTPUT/'colab_logs';logs.mkdir(parents=True,exist_ok=True)
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path=logs/f'{name}_{stamp}.log'
    print('+',shlex.join(command),'\nDurable log:',path,flush=True)
    tail=deque(maxlen=35)
    with path.open('w') as log:
        with subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
            status=process.wait()
        log.write(f'\nEXIT CODE: {status}\n');log.flush()
    if status: raise RuntimeError(f'Exit {status}; full log={path}\n'+'\n'.join(tail))
    return path
print(REVISION,'\nRun:',OUTPUT,'\nNo dataset staging and no GPU required.')


## 2. Install prototype dependencies and run tests
This uses the pinned Urban package version in fresh subprocesses. Existing PyTorch is recorded, not replaced or matched to an old experiment. No CUDA compiler or custom extension is needed.

In [ ]:
run_logged([sys.executable,'-m','pip','install','-q','ultralytics==8.4.61','onnx==1.20.1','onnxruntime==1.30.0'],'dependencies')
run_logged([sys.executable,'-m','unittest','discover','-s','tests','-p','test_parking_shared_model.py','-v'],'tests')


## 3. Build, export and benchmark the shared network
The source/checkpoint download is small compared with dataset staging. The pretrained Urban path runs once per image and feeds both branches. Its weights and BatchNorm statistics are frozen. A synthetic backward check takes **zero optimizer steps**, and synthetic depth-head BatchNorm updates are discarded before export.

The graph returns product candidates plus a quarter-resolution **untrained** depth map. Candidate class filtering is a new explicit prototype policy, not a claim that final detections have the same accuracy as the existing app. Neither preprocessing nor the smaller input resolution has been accuracy-qualified. Host timing includes fetching both graph outputs, but excludes preprocessing/decoding/camera/rendering and is not an iPhone timing.

If this cell fails, preserve its log and still run section 4 to download diagnostics.

In [ ]:
run_logged([sys.executable,'-X','faulthandler','-u',SCRIPT,'--config',CONFIG,'--output',OUTPUT,'--source',SOURCE,'--weights',WEIGHTS],'shared_runtime_screen')
REPORT=OUTPUT/'shared_model_screen.json'
report=json.loads(REPORT.read_text())
assert report['complete'] and report['shared_backbone_and_neck_passes_per_image']==1
assert report['gradient_smoke']['pretrained_weights_and_BN_unchanged']
assert report['depth_head_untrained'] and not report['depth_head_usable_for_measurement']
assert report['optimizer_steps']==0 and not report['accuracy_evaluated']
print(json.dumps({k:report[k] for k in ['parameters','onnx_bytes','host_onnx_cpu_profile','output_bytes_per_image','depth_head_untrained','iphone_performance_measured']},indent=2))


## 4. Download the runtime package or failure diagnostics
Return `parking_shared_runtime_results.zip`. We review the technical screen before adding a separate camera-OFF iPhone benchmark. No phone connection or live-camera test is required yet. A successful export does not authorize range measurement or training on the held-out validation frames.

In [ ]:
bundle=OUTPUT/'parking_shared_runtime_results.zip'
if not bundle.is_file():
    bundle=OUTPUT/'parking_shared_failure_diagnostics.zip'
    with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix in ['.json','.log'] and 'runtime_settings' not in path.parts:
                archive.write(path,path.relative_to(OUTPUT).as_posix())
    print('No successful runtime package: returning diagnostics only.')
print('Download:',bundle)
files.download(str(bundle))
